# Daily Scanner + Stockbee

**Run:** Pre-market or after market close

**Scans:** Bearish Momentum · VCP · Satish · Hardik · NR6 · Rocket Base · EP Breakout · 20d/2m High · Gap Up · Up 4%+ · Stockbee (5 scans)

**Universes:** S&P 500 · Russell 2000 · Nifty 500 (toggle with `INCLUDE_NIFTY`)

**Publishes to:** https://docs.google.com/spreadsheets/d/1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs

**Runtime > Run all**

In [ ]:
# CELL 1 — Install
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q",
    "yfinance","tqdm","requests","lxml","gspread","gspread-dataframe"])
print("Ready")

Ready


In [ ]:
# CELL 2 — Settings
import warnings; warnings.filterwarnings("ignore")
import requests, io, time, gc
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime

INCLUDE_NIFTY         = True    # False = skip Nifty 500
BATCH_SIZE            = 40
SLEEP_BETWEEN_BATCHES = 3
SHEET_ID = "1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs"
print(f"Nifty included: {INCLUDE_NIFTY}")
print(f"Sheet: https://docs.google.com/spreadsheets/d/{SHEET_ID}")

Nifty included: True
Sheet: https://docs.google.com/spreadsheets/d/1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs


In [ ]:
# CELL 3 — Load tickers
HEADERS = {"User-Agent":"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

def get_sp500():
    try:
        html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",headers=HEADERS,timeout=15).text
        t = [str(x).replace(".","-") for x in pd.read_html(io.StringIO(html))[0]["Symbol"].tolist()]
        print(f"  S&P 500      : {len(t)}"); return t
    except Exception as e:
        print(f"  SP500 failed: {e}"); return []

def get_russell2000():
    try:
        nyse = requests.get("https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nyse/nyse_tickers.txt",headers=HEADERS,timeout=15).text.strip().split()
        nasd = requests.get("https://raw.githubusercontent.com/rreichel3/US-Stock-Symbols/main/nasdaq/nasdaq_tickers.txt",headers=HEADERS,timeout=15).text.strip().split()
        sp_csv = requests.get("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv",headers=HEADERS,timeout=15).text
        sp_set = set(pd.read_csv(io.StringIO(sp_csv))["Symbol"].str.replace(".","-",regex=False).tolist())
        all_t = list(dict.fromkeys(nyse+nasd))
        t = [x for x in all_t if x.isalpha() and 2<=len(x)<=4 and x not in sp_set][:2000]
        print(f"  Russell 2000 : {len(t)}"); return t
    except Exception as e:
        print(f"  R2K failed: {e}"); return []

def get_nifty500():
    symbols = [
        "360ONE","3MINDIA","ABB","ACC","ACMESOLAR","AIAENG","APLAPOLLO","AUBANK",
        "AWL","AADHARHFC","AARTIIND","AAVAS","ABBOTINDIA","ACE","ACUTAAS","ADANIENSOL",
        "ADANIENT","ADANIGREEN","ADANIPORTS","ADANIPOWER","ATGL","ABCAPITAL","ABFRL","ABLBL",
        "ABREL","ABSLAMC","CPPLUS","AEGISLOG","AEGISVOPAK","AFCONS","AFFLE","AJANTPHARM",
        "ALKEM","ABDL","AMBER","AMBUJACEM","ANANDRATHI","ANANTRAJ","ANGELONE","ANTHEM",
        "ANURAS","APARINDS","APOLLOHOSP","APOLLOTYRE","APTUS","ASAHIINDIA","ASHOKLEY","ASIANPAINT",
        "ASTERDM","ASTRAL","ATHERENERG","ATUL","AUROPHARMA","AIIL","DMART","AXISBANK",
        "BEML","BLS","BSE","BAJAJ-AUTO","BAJFINANCE","BAJAJFINSV","BAJAJHLDNG","BAJAJHFL",
        "BALKRISIND","BALRAMCHIN","BANDHANBNK","BANKBARODA","BANKINDIA","MAHABANK","BATAINDIA","BAYERCROP",
        "BELRISE","BERGEPAINT","BDL","BEL","BHARATFORG","BHEL","BPCL","BHARTIARTL",
        "BHARTIHEXA","BIKAJI","GROWW","BIOCON","BSOFT","BLUEDART","BLUEJET","BLUESTARCO",
        "BBTC","BOSCHLTD","FIRSTCRY","BRIGADE","BRITANNIA","MAPMYINDIA","CCL","CESC",
        "CGPOWER","CIEINDIA","CRISIL","CANFINHOME","CANBK","CANHLIFE","CAPLIPOINT","CGCL",
        "CARBORUNIV","CARTRADE","CASTROLIND","CEATLTD","CEMPRO","CENTRALBK","CDSL","CHALET",
        "CHAMBLFERT","CHENNPETRO","CHOICEIN","CHOLAHLDNG","CHOLAFIN","CIPLA","CUB","CLEAN",
        "COALINDIA","COCHINSHIP","COFORGE","COHANCE","COLPAL","CAMS","CONCORDBIO","CONCOR",
        "COROMANDEL","CRAFTSMAN","CREDITACC","CROMPTON","CUMMINSIND","CYIENT","DCMSHRIRAM","DLF",
        "DOMS","DABUR","DALBHARAT","DATAPATTNS","DEEPAKFERT","DEEPAKNTR","DELHIVERY","DEVYANI",
        "DIVISLAB","DIXON","LALPATHLAB","DRREDDY","EIDPARRY","EIHOTEL","EICHERMOT","ELECON",
        "ELGIEQUIP","EMAMILTD","EMCURE","EMMVEE","ENDURANCE","ENGINERSIN","ERIS","ESCORTS",
        "ETERNAL","EXIDEIND","NYKAA","FEDERALBNK","FACT","FINCABLES","FSL","FIVESTAR",
        "FORCEMOT","FORTIS","GAIL","GMRAIRPORT","GABRIEL","GALLANTT","GRSE","GICRE",
        "GILLETTE","GLAND","GLAXO","GLENMARK","MEDANTA","GODIGIT","GPIL","GODFRYPHLP",
        "GODREJCP","GODREJIND","GODREJPROP","GRANULES","GRAPHITE","GRASIM","GRAVITA","GESHIP",
        "FLUOROCHEM","GMDCLTD","HEG","HBLENGINE","HCLTECH","HDBFS","HDFCAMC","HDFCBANK",
        "HDFCLIFE","HFCL","HAVELLS","HEROMOTOCO","HEXT","HSCL","HINDALCO","HAL",
        "HINDCOPPER","HINDPETRO","HINDUNILVR","HINDZINC","POWERINDIA","HOMEFIRST","HONASA","HONAUT",
        "HUDCO","HYUNDAI","ICICIBANK","ICICIGI","ICICIAMC","ICICIPRULI","IDBI","IDFCFIRSTB",
        "IFCI","IIFL","IRB","IRCON","ITCHOTELS","ITC","ITI","INDGN",
        "INDIACEM","INDIAMART","INDIANB","IEX","INDHOTEL","IOC","IOB","IRCTC",
        "IRFC","IREDA","IGL","INDUSTOWER","INDUSINDBK","NAUKRI","INFY","INOXWIND",
        "INTELLECT","INDIGO","IGIL","IKS","IPCALAB","JBCHEPHARM","JKCEMENT","JBMA",
        "JKTYRE","JMFINANCIL","JSWCEMENT","JSWDULUX","JSWENERGY","JSWINFRA","JSWSTEEL","JAINREC",
        "JPPOWER","JINDALSAW","JSL","JINDALSTEL","JIOFIN","JUBLFOOD","JUBLINGREA","JUBLPHARMA",
        "JWL","JYOTICNC","KPRMILL","KEI","KPITTECH","KAJARIACER","KPIL","KALYANKJIL",
        "KARURVYSYA","KAYNES","KEC","KFINTECH","KIRLOSENG","KOTAKBANK","KIMS","LTF",
        "LTTS","LGEINDIA","LICHSGFIN","LTFOODS","LTM","LT","LATENTVIEW","LAURUSLABS",
        "THELEELA","LEMONTREE","LENSKART","LICI","LINDEINDIA","LLOYDSME","LODHA","LUPIN",
        "MMTC","MRF","MGL","M&MFIN","M&M","MANAPPURAM","MRPL","MANKIND",
        "MARICO","MARUTI","MFSL","MAXHEALTH","MAZDOCK","MEESHO","MINDACORP","MSUMI",
        "MOTILALOFS","MPHASIS","MCX","MUTHOOTFIN","NATCOPHARM","NBCC","NCC","NHPC",
        "NLCINDIA","NMDC","NSLNISP","NTPCGREEN","NTPC","NH","NATIONALUM","NAVA",
        "NAVINFLUOR","NESTLEIND","NETWEB","NEULANDLAB","NEWGEN","NAM-INDIA","NIVABUPA","NUVAMA",
        "NUVOCO","OBEROIRLTY","ONGC","OIL","OLAELEC","OLECTRA","PAYTM","ONESOURCE",
        "OFSS","POLICYBZR","PCBL","PGEL","PIIND","PNBHOUSING","PTCIL","PVRINOX",
        "PAGEIND","PARADEEP","PATANJALI","PERSISTENT","PETRONET","PFIZER","PHOENIXLTD","PWL",
        "PIDILITIND","PINELABS","PIRAMALFIN","PPLPHARMA","POLYMED","POLYCAB","POONAWALLA","PFC",
        "POWERGRID","PREMIERENE","PRESTIGE","PNB","RRKABEL","RBLBANK","RECLTD","RHIM",
        "RITES","RADICO","RVNL","RAILTEL","RAINBOW","RKFORGE","REDINGTON","RELIANCE",
        "RPOWER","SBFC","SBICARD","SBILIFE","SJVN","SRF","SAGILITY","SAILIFE",
        "SAMMAANCAP","MOTHERSON","SAPPHIRE","SARDAEN","SAREGAMA","SCHAEFFLER","SCHNEIDER","SCI",
        "SHREECEM","SHRIRAMFIN","SHYAMMETL","ENRIN","SIEMENS","SIGNATURE","SOBHA","SOLARINDS",
        "SONACOMS","SONATSOFTW","STARHEALTH","SBIN","SAIL","SUMICHEM","SUNPHARMA","SUNTV",
        "SUNDARMFIN","SUPREMEIND","SPLPETRO","SUZLON","SWANCORP","SWIGGY","SYNGENE","SYRMA",
        "TBOTEK","TVSMOTOR","TATACAP","TATACHEM","TATACOMM","TCS","TATACONSUM","TATAELXSI",
        "TATAINVEST","TMCV","TMPV","TATAPOWER","TATASTEEL","TATATECH","TTML","TECHM",
        "TECHNOE","TEGA","TEJASNET","TENNIND","NIACL","RAMCOCEM","THERMAX","TIMKEN",
        "TITAGARH","TITAN","TORNTPHARM","TORNTPOWER","TARIL","TRAVELFOOD","TRENT","TRIDENT",
        "TRITURBINE","TIINDIA","UCOBANK","UNOMINDA","UPL","UTIAMC","ULTRACEMCO","UNIONBANK",
        "UBL","UNITDSPR","URBANCO","USHAMART","VTL","VBL","VEDL","VIJAYA",
    ]
    t = [s+".NS" for s in list(dict.fromkeys(symbols))]
    print(f"  Nifty 500    : {len(t)}"); return t

print("Loading tickers...")
sp500      = get_sp500()
r2000      = get_russell2000()
r2000_only = [t for t in r2000 if t not in set(sp500)]
nifty500   = get_nifty500() if INCLUDE_NIFTY else []
print(f"  Total        : {len(sp500)+len(r2000_only)+len(nifty500)}")

Loading tickers...
  S&P 500      : 503
  Russell 2000 : 2000
  Nifty 500    : 480
  Total        : 2983


In [ ]:
# CELL 4 — Indicators
import pandas as pd
import numpy as np

def sma(s,n): return s.rolling(n).mean()
def ema(s,n): return s.ewm(span=n,adjust=False).mean()
def rsi(s,n=14):
    d=s.diff()
    g=d.clip(lower=0).rolling(n).mean()
    l=(-d.clip(upper=0)).rolling(n).mean()
    return 100-(100/(1+g/l.replace(0,np.nan)))
def _f(x):
    try: return float(x)
    except: return float("nan")
def _range(d,i):
    try: return _f(d["High"].iloc[-(i+1)]) - _f(d["Low"].iloc[-(i+1)])
    except: return float("nan")
def vwap_intraday(h):
    h=h.copy()
    h["_d"]=h.index.normalize()
    h["_tp"]=(h["High"]+h["Low"]+h["Close"])/3
    tpv=h.groupby("_d").apply(lambda g:(g["_tp"]*g["Volume"]).cumsum()).values
    cvol=h.groupby("_d")["Volume"].cumsum().values
    return pd.Series(tpv/cvol,index=h.index)
print("Indicators ready")

Indicators ready


In [ ]:
# CELL 5 — Fetchers (daily/weekly + intraday separated)
import yfinance as yf
import pandas as pd
import time

def fetch_batch(tickers, period, interval, retries=2):
    """Daily / weekly batch fetcher."""
    out = {}
    if not tickers: return out
    for attempt in range(retries+1):
        try:
            raw = yf.download(tickers, period=period, interval=interval,
                              group_by="ticker", auto_adjust=False,
                              progress=False, threads=True)
            if raw.empty: break
            if len(tickers)==1:
                t=tickers[0]; df=raw.copy()
                if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(1)
                df=df.drop(columns=["Adj Close"],errors="ignore")
                df.dropna(how="all",inplace=True)
                if len(df)>5: out[t]=df
            else:
                for t in tickers:
                    try:
                        if t in raw.columns.levels[0]:
                            df=raw[t].copy()
                            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(-1)
                            df=df.drop(columns=["Adj Close"],errors="ignore")
                            df=df.dropna(how="all")
                            if len(df)>5: out[t]=df
                    except: pass
            break
        except Exception as e:
            if any(x in str(e) for x in ["Rate","429","Too Many","RateLimit"]):
                wait=30*(attempt+1); print(f"  Rate limit — wait {wait}s"); time.sleep(wait)
            else: break
    return out

def fetch_intraday(tickers, period="5d", interval="1h", retries=2):
    """Intraday fetcher (1h) — uses your working MultiIndex fix."""
    out = {}
    if not tickers: return out
    for attempt in range(retries+1):
        try:
            raw = yf.download(tickers, period=period, interval=interval,
                              group_by="ticker", auto_adjust=False,
                              progress=False, threads=True)
            if raw.empty: break
            if len(tickers)==1:
                t=tickers[0]; df=raw.copy()
                if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(1)
                df=df.drop(columns=["Adj Close"],errors="ignore")
                df.dropna(how="all",inplace=True)
                if len(df)>5: out[t]=df
            else:
                for t in tickers:
                    try:
                        if t in raw.columns.levels[0]:
                            df=raw[t].copy()
                            if isinstance(df.columns,pd.MultiIndex): df.columns=df.columns.get_level_values(-1)
                            df=df.drop(columns=["Adj Close"],errors="ignore")
                            df=df.dropna(how="all")
                            if len(df)>5: out[t]=df
                    except: pass
            break
        except Exception as e:
            if any(x in str(e) for x in ["Rate","429","Too Many","RateLimit"]):
                wait=30*(attempt+1); print(f"  Rate limit — wait {wait}s"); time.sleep(wait)
            else: break
    return out

print("fetch_batch (daily/weekly) + fetch_intraday (1h) ready")

fetch_batch (daily/weekly) + fetch_intraday (1h) ready


In [ ]:
# CELL 6 — Daily scan functions
import numpy as np

def scan_bearish_momentum(d):
    try:
        if len(d)<52: return False
        c=_f(d["Close"].iloc[-1])
        return (c>50 and _f(rsi(d["Close"]).iloc[-1])<50
            and c<_f(d["Low"].iloc[-2])
            and c<_f(sma(d["Close"],50).iloc[-1])
            and _f(sma(d["Volume"],20).iloc[-1])>500000)
    except: return False

def scan_vcp(d, mktcap_m):
    try:
        if len(d)<252: return False
        c=_f(d["Close"].iloc[-1]); dvol=c*_f(sma(d["Volume"],20).iloc[-1])
        s200=_f(sma(d["Close"],200).iloc[-1]); s50=_f(sma(d["Close"],50).iloc[-1])
        c22=_f(d["Close"].iloc[-23]); c66=_f(d["Close"].iloc[-67])
        hi=_f(d["High"].rolling(252).max().iloc[-1])
        return ((c/c22>1.2 and mktcap_m>1 and dvol>30e6 and c>s200) or
                (c/c66>=1.3 and c>=1 and dvol>30e6 and c>s200) or
                (mktcap_m>=1000 and c>hi*0.75 and c>s50 and c>s200 and dvol>30e6))
    except: return False

def scan_satish_bullish(d, mktcap_m):
    try:
        if len(d)<47: return False
        c=_f(d["Close"].iloc[-1]); o=_f(d["Open"].iloc[-1])
        lo=_f(d["Low"].iloc[-1]); v=_f(d["Volume"].iloc[-1])
        pc=_f(d["Close"].iloc[-2]); pl=_f(d["Low"].iloc[-2]); ph=_f(d["High"].iloc[-2])
        pivot=(ph+pl+pc)/3; r=rsi(d["Close"])
        return (mktcap_m>=1000 and c>=100 and c>pivot
            and v>_f(sma(d["Volume"],10).iloc[-1])
            and lo>pl and c>pc and c>o
            and _f(sma(d["Close"],9).iloc[-1])>_f(ema(d["Close"],45).iloc[-1])
            and _f(r.iloc[-3])>_f(r.iloc[-2])
            and (c-pc)/pc*100>1 and _f(r.iloc[-1])>30 and v>100000)
    except: return False

def scan_hardik(d, w, mktcap_m, is_nifty=False):
    # mktcap threshold: 5000 crores for Nifty (~$600M), 5000 USD millions for US ($5B)
    try:
        if len(d)<12 or len(w)<16: return False
        return (_f(rsi(w["Close"]).iloc[-1])>50
            and _f(d["Close"].iloc[-1])>_f(sma(d["Close"],10).iloc[-1])
            and _f(sma(d["Close"],10).iloc[-6])>_f(d["Close"].iloc[-6])
            and _f(d["Close"].iloc[-2])<_f(d["Open"].iloc[-2])
            and _f(d["Close"].iloc[-1])>_f(d["Open"].iloc[-1])
            and mktcap_m>5000)
    except: return False

def scan_nr6(d):
    try:
        if len(d)<8: return False
        r0=_range(d,0)
        return all(r0<_range(d,i) for i in range(1,7))
    except: return False

def scan_rocket_base(d):
    try:
        if len(d)<92: return False
        c=_f(d["Close"].iloc[-1])
        if c<=30 or _f(sma(d["Volume"],50).iloc[-1])<50000: return False
        return (c>=_f(d["Low"].iloc[-6])*1.2 or
                c>=_f(d["Low"].iloc[-31])*1.3 or
                c>=_f(d["Low"].iloc[-91])*1.3)
    except: return False

def scan_ep_breakout(d):
    try:
        if len(d)<130: return False
        max5=_f(d["Close"].rolling(5).max().iloc[-1])
        max120_6ago=_f(d["Close"].rolling(120).max().iloc[-7])
        return (max5>max120_6ago*1.05
            and _f(d["Volume"].iloc[-1])>_f(sma(d["Volume"],5).iloc[-1])
            and _f(d["Close"].iloc[-1])>_f(d["Close"].iloc[-2]))
    except: return False

def scan_20day_high(d, is_us=True):
    if not is_us: return False
    try:
        if len(d)<21: return False
        return _f(d["Close"].iloc[-1])>=_f(d["High"].rolling(20).max().iloc[-1])
    except: return False

def scan_2month_high(d, is_us=True):
    if not is_us: return False
    try:
        if len(d)<43: return False
        return _f(d["Close"].iloc[-1])>=_f(d["High"].rolling(42).max().iloc[-1])
    except: return False

def scan_gapup(d):
    try:
        if len(d)<2: return False
        return _f(d["Open"].iloc[-1])>=_f(d["Close"].iloc[-2])*1.03
    except: return False

def scan_up4pct(d):
    try:
        if len(d)<2: return False
        c=_f(d["Close"].iloc[-1]); o=_f(d["Open"].iloc[-1])
        return (c-o)/o>=0.04
    except: return False

print("Daily scan functions ready (11 scans)")

Daily scan functions ready (11 scans)


In [ ]:
# CELL 7 — Daily scan runner
import time

DAILY_SCAN_KEYS = [
    "bearish_momentum","vcp_setup","satish_bullish","hardik",
    "nr6","rocket_base","ep_breakout","high_20d","high_2month",
    "gapup_3pct","up_4pct"
]

def run_daily_scans(tickers, label):
    is_nifty = label=="Nifty 500"
    is_us    = not is_nifty
    hits = {k:[] for k in DAILY_SCAN_KEYS}
    batches=[tickers[i:i+BATCH_SIZE] for i in range(0,len(tickers),BATCH_SIZE)]
    total=len(batches)
    print(f"Scanning {label} — {len(tickers)} tickers, {total} batches")
    for n,batch in enumerate(batches):
        print(f"  Batch {n+1}/{total}",end="\r")
        daily  = fetch_batch(batch,"2y","1d")
        weekly = fetch_batch(batch,"5y","1wk")
        for t in batch:
            d=daily.get(t); w=weekly.get(t)
            if d is None or len(d)<10: continue
            try:
                c=float(d["Close"].iloc[-1]); av=float(sma(d["Volume"],20).iloc[-1])
                mktcap = c*av*30/(1e7 if is_nifty else 1e6)
            except: mktcap=0
            if scan_bearish_momentum(d):                     hits["bearish_momentum"].append(t)
            if scan_vcp(d,mktcap):                           hits["vcp_setup"].append(t)
            if scan_satish_bullish(d,mktcap):                hits["satish_bullish"].append(t)
            if w is not None and scan_hardik(d,w,mktcap,is_nifty): hits["hardik"].append(t)
            if scan_nr6(d):                                  hits["nr6"].append(t)
            if scan_rocket_base(d):                          hits["rocket_base"].append(t)
            if scan_ep_breakout(d):                          hits["ep_breakout"].append(t)
            if scan_20day_high(d,is_us):                     hits["high_20d"].append(t)
            if scan_2month_high(d,is_us):                    hits["high_2month"].append(t)
            if scan_gapup(d):                                hits["gapup_3pct"].append(t)
            if scan_up4pct(d):                               hits["up_4pct"].append(t)
        time.sleep(SLEEP_BETWEEN_BATCHES)
    print(f"\n{label} done")
    return hits

print("Daily runner ready")

Daily runner ready


In [ ]:
# CELL 8 — RUN DAILY SCANS
# S&P 500: ~10 min | Russell 2000: ~20 min | Nifty: ~10 min
from datetime import datetime
start=datetime.now()
print(f"Started: {start.strftime('%H:%M:%S')}")
sp500_hits  = run_daily_scans(sp500,      "S&P 500")
r2000_hits  = run_daily_scans(r2000_only, "Russell 2000")
nifty_hits  = run_daily_scans(nifty500,   "Nifty 500") if INCLUDE_NIFTY else {k:[] for k in DAILY_SCAN_KEYS}
print(f"Done in ~{(datetime.now()-start).seconds//60} min")

# ── Save to Google Drive for Streamlit website ──────────────────
import os, pandas as pd
from datetime import datetime

# Mount Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _save_dir = '/content/drive/MyDrive/Stockbee'
    os.makedirs(_save_dir, exist_ok=True)
except Exception:
    _save_dir = '.'

# Build flat DataFrame from hits dicts
_ts  = datetime.now().strftime('%Y-%m-%d %H:%M')
_rows = []
for _k in DAILY_SCAN_KEYS:
    _label = {"bearish_momentum":"Bearish Momentum","vcp_setup":"VCP Setup","satish_bullish":"Satish Bullish","hardik":"Hardik Scan","nr6":"NR6 Narrowest Range","rocket_base":"Rocket Base","ep_breakout":"EP Breakout","high_20d":"20-Day High (US)","high_2month":"2-Month High (US)","gapup_3pct":"Gap Up 3pct","up_4pct":"Up 4pct"}.get(_k, _k)
    for _t in sp500_hits.get(_k, []):  _rows.append({'Scan':_label,'Ticker':_t,'Universe':'S&P 500',  'Updated':_ts})
    for _t in r2000_hits.get(_k, []): _rows.append({'Scan':_label,'Ticker':_t,'Universe':'Russell 2000','Updated':_ts})
    for _t in nifty_hits.get(_k, []):  _rows.append({'Scan':_label,'Ticker':_t,'Universe':'Nifty 500', 'Updated':_ts})

_daily_df = pd.DataFrame(_rows) if _rows else pd.DataFrame(columns=['Scan','Ticker','Universe','Updated'])
_csv_path = os.path.join(_save_dir, 'daily_hits.csv')
_daily_df.to_csv(_csv_path, index=False)
print(f"\n✅ Saved daily_hits.csv → {_csv_path}  ({len(_daily_df)} rows)")
print(f"   Scans: {_daily_df['Scan'].nunique() if not _daily_df.empty else 0}  |  Tickers: {_daily_df['Ticker'].nunique() if not _daily_df.empty else 0}")


Started: 16:15:21
Scanning S&P 500 — 503 tickers, 13 batches

S&P 500 done
Scanning Russell 2000 — 2000 tickers, 50 batches


ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AHL"}}}
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['AKO']: YFPricesMissingError('possibly delisted; no price data found  (period=2y)')
ERROR:yfinance:['AHL']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: AHL"}}}
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['AKO']: YFPricesMissingError('possibly delisted; no price data found  (period=5y)')
ERROR:yfinance:['AHL']: YFPricesMissingError('possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")')


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ANG']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ANG']: YFPricesMissingError('possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")')


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ATH']: YFPricesMissingError('possibly delisted; no price data found  (period=2y) (Yahoo error = "No data found, symbol may be delisted")')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['ATH']: YFPricesMissingError('possibly delisted; no price data found  (period=5y) (Yahoo error = "No data found, symbol may be delisted")')


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BF']: YFPricesMissingError('possibly delisted; no price data found  (period=2y)')
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['BF']: YFPricesMissingError('possibly delisted; no price data found  (period=5y)')


...

ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['GLAXO.NS', 'HCLTECH.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['JBMA.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['MAXHEALTH.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['TATATECH.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')



Nifty 500 done
Done in ~13 min


In [ ]:
# CELL 9 — Print summary
from datetime import datetime
LABELS={"bearish_momentum":"Bearish Momentum","vcp_setup":"VCP Setup",
    "satish_bullish":"Satish Bullish","hardik":"Hardik Scan",
    "nr6":"NR6 Narrowest Range","rocket_base":"Rocket Base",
    "ep_breakout":"EP Breakout","high_20d":"20-Day High (US)",
    "high_2month":"2-Month High (US)","gapup_3pct":"Gap Up 3pct","up_4pct":"Up 4pct"}
print("\n"+"="*65)
print(f"  DAILY RESULTS  {datetime.now().strftime('%d %b %Y  %H:%M')}")
print("="*65)
for k,label in LABELS.items():
    sp=sp500_hits.get(k,[]); r2k=r2000_hits.get(k,[]); nif=nifty_hits.get(k,[])
    tot=len(sp)+len(r2k)+len(nif)
    if tot==0: continue
    print(f"  {label:<38} S&P:{len(sp):>3} R2K:{len(r2k):>3} Nifty:{len(nif):>3} Total:{tot:>4}")
    if sp:  print(f"    S&P:   {chr(32).join(sp[:15])}")
    if r2k: print(f"    R2K:   {chr(32).join(r2k[:15])}")
    if nif: print(f"    Nifty: {chr(32).join(nif[:15])}")
print("="*65)


  DAILY RESULTS  21 Jul 2026  16:28
  Bearish Momentum                       S&P: 35 R2K: 13 Nifty: 18 Total:  66
    S&P:   LNT GOOGL GOOG AEE AEP BLDR CHD CMS CL STZ COST CRH DHR DTE EW
    R2K:   CAVA DKS DTM FERG LPX LRN MHK NGG SUI SUNB SXT TWLO WMS
    Nifty: ATGL.NS AFCONS.NS ANGELONE.NS FIRSTCRY.NS DRREDDY.NS ELECON.NS EMAMILTD.NS HDFCAMC.NS HDFCBANK.NS HUDCO.NS ITCHOTELS.NS POLICYBZR.NS RELIANCE.NS SJVN.NS SCI.NS
  VCP Setup                              S&P:239 R2K:345 Nifty:170 Total: 754
    S&P:   MMM ABBV AMD AES AFL A APD ABNB AKAM ALGN ALL MO AMCR AXP AIG
    R2K:   AAMI ABCB ACA ADC AEG AER AFG AGL AHR AIR AIT AKR ALLY AM AMC
    Nifty: 360ONE.NS ABB.NS ACMESOLAR.NS AIAENG.NS AARTIIND.NS ABBOTINDIA.NS ACE.NS ACUTAAS.NS ADANIENSOL.NS ADANIENT.NS ADANIGREEN.NS ADANIPORTS.NS ABCAPITAL.NS AEGISLOG.NS AEGISVOPAK.NS
  Satish Bullish                         S&P:  0 R2K:  1 Nifty: 19 Total:  20
    R2K:   CCK
    Nifty: AWL.NS ACE.NS ABCAPITAL.NS BAJAJFINSV.NS BAJAJHLDNG.NS BA

---
## Stockbee US Screener
Runs after daily scans. Results added to the same Google Sheet.

In [ ]:
# Stockbee: Install & Imports
# Most packages already installed above. Just ensure tqdm is present.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm'])

import warnings, os, gc, time
from datetime import datetime
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
print('✅ Stockbee imports ready')


In [ ]:
# Stockbee: Configuration
# Aligned to Pine Script v2 — DAMA removed, 9M Vol / +4% / -4% updated

# ── Universe pre-filters ──
MIN_PRICE           = 10.0   # close >= $10
MIN_AVG_VOL_50K     = 100    # 50d avg vol >= 100K (scans 1/2/3)
MIN_AVG_VOL_200K    = 200    # 50d avg vol >= 200K (scan 4)

# ── EP Scan thresholds ──
EXTREME_SALES_PCT   = 99     # Scan 1: revenue YoY >= 99%
SALES_GROWTH_PCT    = 39     # Scan 2/3: revenue YoY >= 39% (last Q AND 2Q avg)
MIN_ANNUAL_SALES_M  = 25     # trailing 4Q revenue >= $25M
EPS_GROWTH_PCT      = 39     # Scan 4: EPS YoY >= 39% (BOTH quarters required)
SALES_GROWTH_EP4    = 20     # Scan 4: revenue YoY >= 20%
MKTCAP_SCAN3_B      = 11     # Scan 3: mkt cap <= $11B (USD only)
IPO_YEARS_SCAN3     = 10     # Scan 3: IPO within 10 years

# ── Entry MA & ATR (DAMA removed in Pine v2) ──
ENTRY_MA_PERIOD     = 70     # MA70 for ATR entry threshold
ATR_MULTIPLE        = 2.0    # price >= MA + N*ATR to be in entry zone
USE_DI_FILTER       = True   # require +DI > -DI (price > MA proxy)

# ── Pattern filter toggles ──
USE_TTT_FILTER      = False
USE_MWG_FILTER      = False
USE_BMS_FILTER      = False

# ── Momentum flags (Pine v2 updated definitions) ──
TI65_BULL           = 1.05   # avgC7 / avgC65 >= 1.05
VOL_9M              = 9_000_000  # 9M Vol: also needs vol>prior AND close up 4%+
PLUS4PCT            = 4.0    # threshold for BO and BD flags

# ── Gap detection (TTT & MWG shared) ──
GAP_JUMP_MULT       = 1.20
GAP_RANGE_PCT       = 0.04
GAP_LOOKBACK        = 100

# ── Performance ──
SB_MAX_WORKERS      = 8      # parallel threads
SB_BATCH_SIZE       = 50     # checkpoint every N tickers
SB_HISTORY          = '18mo'

# ── Output files ──
RESULTS_FILE        = 'stockbee_results.csv'
HITS_FILE           = 'stockbee_hits.csv'

print('✅ Stockbee config loaded (Pine v2 aligned)')


In [ ]:
# Stockbee: Universe Builder

def fetch_us_tickers():
    SKIP_SUFFIXES = ('W', 'R', 'U', 'Z', 'L')
    KNOWN_ETFS    = {'SPY','QQQ','IWM','DIA','GLD','SLV','TLT','HYG','LQD',
                     'XLB','XLC','XLE','XLF','XLI','XLK','XLP','XLU','XLV','XLY'}
    url = ('https://raw.githubusercontent.com/'
           'rreichel3/US-Stock-Symbols/main/all/all_tickers.txt')
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        raw = [t.strip() for t in resp.text.split('\n') if t.strip()]
    except Exception as e:
        print(f'⚠️  Ticker list failed ({e}). Using fallback.')
        return _sb_fallback_tickers()
    clean = []
    for t in raw:
        if len(t) > 5:                          continue
        if '.' in t or '-' in t:               continue
        if t in KNOWN_ETFS:                     continue
        if t.endswith(SKIP_SUFFIXES) and len(t) > 3: continue
        clean.append(t)
    print(f'✅ Stockbee universe: {len(clean):,} US tickers')
    return clean

def _sb_fallback_tickers():
    return [
        'AAPL','MSFT','NVDA','TSLA','AMZN','META','GOOGL','AMD','AVGO','NFLX',
        'SMCI','PLTR','MSTR','IONQ','CRWD','SNOW','DDOG','ZS','MDB','BILL',
        'AFRM','UPST','SOFI','HOOD','RBLX','COIN','PATH','AI','GTLB','SOUN',
        'JOBY','ACHR','RKLB','ASTS','LUNR','RDDT','CELH','ELF','HIMS','IMVT',
        'AXSM','INSM','CAVA','BROS','WING','CMG','SHAK','SBUX','SHOP','MELI',
        'REGN','BIIB','VRTX','ALNY','MRNA','BNTX','GILD','AMGN','ABBV','LLY',
        'JPM','BAC','WFC','GS','MARA','RIOT','CLSK','WULF','IREN','COST',
    ]

ALL_TICKERS = fetch_us_tickers()
print(f'   Ready to scan {len(ALL_TICKERS):,} tickers')


In [ ]:
from typing import Optional
# Stockbee: Core Calculation Engine — Pine Script v2 aligned

# ── Revenue YoY: Pine requires prior > 0 (positive base only) ──
def sb_safe_pct_rev(new_val, old_val):
    if pd.isna(new_val) or pd.isna(old_val) or old_val <= 0: return None
    return ((new_val - old_val) / old_val) * 100

# ── EPS YoY: Pine uses abs(prior) as denominator ──
def sb_safe_pct_eps(new_val, old_val):
    if pd.isna(new_val) or pd.isna(old_val) or old_val == 0: return None
    return ((new_val - old_val) / abs(old_val)) * 100

def sb_calc_atr(high, low, close, period=14):
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(alpha=1/period, adjust=False).mean()

def sb_count_gaps(close, high, low, lookback=100):
    """
    Gap bar: close > 1.2*prior AND (high-low) < 0.04*close
    Pine v2 fix: requires prior close is not NaN (not na(close[1]) guard)
    """
    prior = close.shift(1)
    is_gap = (
        prior.notna() &
        (close > GAP_JUMP_MULT * prior) &
        ((high - low) < GAP_RANGE_PCT * close)
    ).astype(int)
    return is_gap.rolling(lookback, min_periods=lookback).sum()

def sb_get_fundamentals(tk):
    out = dict(rev_q=[], eps_q=[], annual_sales_m=None,
               market_cap=None, currency=None, ipo_years_ago=None)
    try:
        fi = tk.fast_info
        out['market_cap'] = getattr(fi, 'market_cap', None)
        out['currency']   = getattr(fi, 'currency', None)
    except Exception: pass
    try:
        fins = tk.quarterly_financials
        if fins is not None and not fins.empty:
            for label in ['Total Revenue', 'TotalRevenue', 'Revenue']:
                if label in fins.index:
                    row = fins.loc[label].sort_index(ascending=False)
                    out['rev_q'] = [v for v in row.values if not pd.isna(v)]
                    break
    except Exception: pass
    try:
        eps_df = tk.quarterly_earnings
        if eps_df is not None and not eps_df.empty and 'Reported EPS' in eps_df.columns:
            eps_s = eps_df.sort_index(ascending=False)['Reported EPS']
            out['eps_q'] = [v for v in eps_s.values if not pd.isna(v)]
    except Exception: pass
    if len(out['rev_q']) >= 4:
        out['annual_sales_m'] = sum(out['rev_q'][:4]) / 1_000_000
    try:
        info = tk.info
        raw  = info.get('firstTradeDateEpochUtc') or info.get('ipoExpectedDate')
        if raw:
            dt = datetime.fromtimestamp(raw) if isinstance(raw, (int,float)) else pd.to_datetime(raw)
            out['ipo_years_ago'] = (datetime.now() - dt).days / 365.25
    except Exception: pass
    return out

# ── Pattern filters ──
def sb_calc_ttt(close, high, low, volume,
                min_vol=300_000, min_price=10.0, max_3bar=1.5, max_today=0.3, lookback=100):
    """Ants TTT: tight consolidation, no gaps, volume (fail-closed)."""
    vol_ok    = volume.shift(1).rolling(3).min() >= min_vol
    price_ok  = close > min_price
    pct_3bar  = ((close - close.shift(3)) / close.shift(3) * 100).abs()
    consol_ok = pct_3bar <= max_3bar
    pct_today = ((close - close.shift(1)) / close.shift(1) * 100).abs()
    tight_ok  = pct_today <= max_today
    no_gaps   = sb_count_gaps(close, high, low, lookback) == 0
    return (vol_ok & price_ok & consol_ok & tight_ok & no_gaps).fillna(False)

def sb_calc_mwg(close, high, low, volume,
                min_vol=100_000, vol_days=3, min_price=10.0, max_day=0.4, lookback=100):
    """Ants Bullish/MWG: momentum without gaps (fail-closed)."""
    low30      = close.rolling(30).min()
    avg7       = close.rolling(7).mean()
    avg65      = close.rolling(65).mean()
    mom_ok     = (close / low30 >= 1.20) | (avg7 / avg65 >= 1.05)
    price_ok   = close > min_price
    ctrl_ok    = ((close - close.shift(1)) / close.shift(1) * 100).abs() <= max_day
    vol_ok     = volume.shift(1).rolling(vol_days).min() >= min_vol
    no_gaps    = sb_count_gaps(close, high, low, lookback) == 0
    return (mom_ok & price_ok & ctrl_ok & vol_ok & no_gaps).fillna(False)

def sb_calc_bms(close, high, low, volume, open_,
                min_price=10.0, min_vol_c1=1_000_000, min_vol_c2=100_000,
                breakout_pct=4.0, stability_pct=2.0, close_str_min=0.70):
    """Bullish Combo: candle pattern + volume. Pine v2 fix: open>0 guard."""
    candle   = ((close - open_) / open_.clip(lower=0.01)) >= 0.02
    today_r  = high - low
    stability = ((close.shift(1) / close.shift(2)) - 1).abs() <= (stability_pct / 100)
    cond1    = candle & (volume > min_vol_c1) & (today_r >= (high.shift(1)-low.shift(1)).abs()) & stability
    cond2    = ((close / close.shift(1)) >= (1 + breakout_pct/100)) & (volume > volume.shift(1)) & (volume >= min_vol_c2) & stability
    price_ok = close >= min_price
    cs_ok    = ((close - low) / (high - low).clip(lower=1e-9)) >= close_str_min
    return ((cond1 | cond2) & price_ok & cs_ok).fillna(False)


# ── Main single-ticker screener ──
def sb_screen_one(sym):
    row = dict(
        ticker=sym, price=None,
        scan1=False, scan2=False, scan3=False, scan4=False, momentum=False,
        ttt_pass=False, mwg_pass=False, bms_pass=False, di_bullish=False,
        flag_ti65=False,
        flag_9mvol=False,        # 9M vol + vol>prior + close up 4%+ (Pine v2)
        flag_plus4_bo=False,     # +4% breakout: pct>=4 AND vol>prior (Pine v2)
        flag_minus4_bd=False,    # -4% breakdown: pct<=-4 AND vol>prior (NEW Pine v2)
        ti65=None, sales_pct=None, eps_pct=None,
        annual_sales_m=None, mktcap_b=None, avg_vol_50k=None,
        adr_pct=None, ext_atr=None, error=None,
    )
    try:
        tk   = yf.Ticker(sym)
        hist = tk.history(period=SB_HISTORY, interval='1d', auto_adjust=True)
        if hist.empty or len(hist) < 70:
            row['error'] = 'no_data'; return row

        close  = hist['Close']
        high   = hist['High']
        low    = hist['Low']
        volume = hist['Volume']
        open_  = hist['Open']

        last_c  = float(close.iloc[-1])
        last_v  = float(volume.iloc[-1])
        avg50k  = float(volume.rolling(50).mean().iloc[-1]) / 1000

        if last_c < MIN_PRICE or avg50k < MIN_AVG_VOL_50K:
            row['error'] = 'filtered'; return row

        row['price']       = round(last_c, 2)
        row['avg_vol_50k'] = round(avg50k, 1)

        atr14 = sb_calc_atr(high, low, close, 14)
        latr  = float(atr14.iloc[-1])
        ema   = close.rolling(ENTRY_MA_PERIOD).mean()
        lma   = float(ema.iloc[-1])

        # ATR entry zone (DAMA removed in Pine v2)
        in_entry_zone = last_c >= lma + ATR_MULTIPLE * latr

        # ADR%
        row['adr_pct'] = round(float(((high-low)/close*100).rolling(14).mean().iloc[-1]), 2)

        # Extension from MA70 in ATR units
        ma70 = close.rolling(70).mean()
        row['ext_atr'] = round((last_c - float(ma70.iloc[-1])) / latr, 2) if latr > 0 else None

        # TI65: avgC7 / avgC65
        avg7   = close.rolling(7).mean()
        avg65  = close.rolling(65).mean()
        lti65  = float((avg7 / avg65).iloc[-1]) if not pd.isna((avg7/avg65).iloc[-1]) else 0
        row['ti65']      = round(lti65, 3)
        row['flag_ti65'] = lti65 >= TI65_BULL

        # DI proxy: price above MA
        row['di_bullish'] = last_c > lma

        # Prior day values
        prior_c = float(close.iloc[-2])  if len(close)  >= 2 else last_c
        prior_v = float(volume.iloc[-2]) if len(volume) >= 2 else last_v
        pct_today = ((last_c - prior_c) / prior_c * 100) if prior_c else 0

        # 9M Vol flag — Pine v2: vol>=9M AND vol>prior AND close>=1.04*prior
        row['flag_9mvol'] = (last_v >= VOL_9M and last_v > prior_v
                             and last_c >= 1.04 * prior_c)

        # +4% Breakout — Pine v2: pct>=4 AND vol>prior
        row['flag_plus4_bo']  = pct_today >= PLUS4PCT and last_v > prior_v

        # -4% Breakdown — NEW Pine v2: pct<=-4 AND vol>prior
        row['flag_minus4_bd'] = pct_today <= -PLUS4PCT and last_v > prior_v

        # Pattern filters
        row['ttt_pass'] = bool(sb_calc_ttt(close, high, low, volume).iloc[-1])
        row['mwg_pass'] = bool(sb_calc_mwg(close, high, low, volume).iloc[-1])
        row['bms_pass'] = bool(sb_calc_bms(close, high, low, volume, open_).iloc[-1])

        passes_di  = row['di_bullish'] if USE_DI_FILTER  else True
        passes_ttt = row['ttt_pass']   if USE_TTT_FILTER else True
        passes_mwg = row['mwg_pass']   if USE_MWG_FILTER else True
        passes_bms = row['bms_pass']   if USE_BMS_FILTER else True

        # Momentum = ATR entry zone + optional filters (DAMA removed)
        row['momentum'] = in_entry_zone and passes_di and passes_ttt and passes_mwg and passes_bms

        # Fundamentals
        fin  = sb_get_fundamentals(tk)
        revq = fin['rev_q']
        epsq = fin['eps_q']
        ann  = fin['annual_sales_m']
        mcap = fin['market_cap']
        curr = fin['currency']
        ipo  = fin['ipo_years_ago']

        row['annual_sales_m'] = round(ann, 1)       if ann  else None
        row['mktcap_b']       = round(mcap/1e9, 2)  if mcap else None

        # Revenue YoY — Pine: prior > 0 (fail-closed)
        sg  = sb_safe_pct_rev(revq[0], revq[4]) if len(revq) >= 5 else None
        sg1 = sb_safe_pct_rev(revq[1], revq[5]) if len(revq) >= 6 else None
        # avgSalesChg2Q: FAIL-CLOSED — both quarters required (Pine v2 change)
        sg2 = (sg + sg1) / 2 if (sg is not None and sg1 is not None) else None
        row['sales_pct'] = round(sg, 1) if sg is not None else None

        # EPS YoY — Pine: abs(prior) denominator
        eg  = sb_safe_pct_eps(epsq[0], epsq[4]) if len(epsq) >= 5 else None
        eg1 = sb_safe_pct_eps(epsq[1], epsq[5]) if len(epsq) >= 6 else None
        row['eps_pct'] = round(eg, 1) if eg is not None else None

        price_ok  = last_c  >= MIN_PRICE
        vol100_ok = avg50k  >= MIN_AVG_VOL_50K
        vol200_ok = avg50k  >= MIN_AVG_VOL_200K
        sales_ok  = ann is not None and ann >= MIN_ANNUAL_SALES_M

        # Scan 1: Extreme Sales (99%+ YoY)
        row['scan1'] = (sg is not None and sg >= EXTREME_SALES_PCT
                        and sales_ok and price_ok and vol100_ok)

        # Scan 2: Sales Growth 39/39 — fail-closed 2Q avg
        sales_core = (sg  is not None and sg  >= SALES_GROWTH_PCT
                      and sg2 is not None and sg2 >= SALES_GROWTH_PCT
                      and sales_ok and price_ok and vol100_ok)
        row['scan2'] = sales_core

        # Scan 3: Growth/Turnaround — USD currency check (Pine v2)
        usd_ok    = (curr == 'USD' or curr is None)
        small_mid = (mcap is not None and mcap <= MKTCAP_SCAN3_B * 1e9 and usd_ok)
        new_ipo   = (ipo is None or ipo <= IPO_YEARS_SCAN3)
        row['scan3'] = sales_core and small_mid and new_ipo

        # Scan 4: EPS + Sales — BOTH eps quarters required (Pine v2)
        sg2_ep4 = (sg + sg1) / 2 if (sg is not None and sg1 is not None) else None
        row['scan4'] = (eg  is not None and eg  >= EPS_GROWTH_PCT
                        and eg1 is not None and eg1 >= EPS_GROWTH_PCT
                        and sg  is not None and sg  >= SALES_GROWTH_EP4
                        and sg2_ep4 is not None and sg2_ep4 >= SALES_GROWTH_EP4
                        and sales_ok and price_ok and vol200_ok)

    except Exception as exc:
        row['error'] = str(exc)[:60]
    return row

print('✅ Stockbee calculation engine ready (Pine v2 aligned)')


In [ ]:
import gc
# Stockbee: Main Scan Runner

def sb_save_checkpoint(results, cp_path, hits_path):
    try:
        df_cp = pd.DataFrame(results)
        scan_cols = [c for c in ['scan1','scan2','scan3','scan4','momentum'] if c in df_cp]
        df_cp['any_hit'] = df_cp[scan_cols].any(axis=1)
        df_cp.to_csv(cp_path, index=False)
        df_cp[df_cp['any_hit']].to_csv(hits_path, index=False)
    except Exception: pass


def run_stockbee_scan(tickers, save_dir='.'):
    cp_path      = os.path.join(save_dir, 'sb_checkpoint.csv')
    hits_path    = os.path.join(save_dir, HITS_FILE)
    results_path = os.path.join(save_dir, RESULTS_FILE)

    # Resume from checkpoint
    results, done = [], set()
    if os.path.exists(cp_path):
        try:
            prev = pd.read_csv(cp_path)
            results = prev.to_dict('records')
            done    = set(prev['ticker'].tolist())
            print(f'♻️  Resuming: {len(done)} already done')
        except Exception: pass

    remaining  = [t for t in tickers if t not in done]
    start_time = time.time()
    print(f'\n{"━"*60}')
    print(f'  STOCKBEE SCAN  |  {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'  {len(remaining):,} tickers  |  {SB_MAX_WORKERS} threads')
    print(f'{"━"*60}\n')

    with ThreadPoolExecutor(max_workers=SB_MAX_WORKERS) as pool:
        futures = {pool.submit(sb_screen_one, sym): sym for sym in remaining}
        with tqdm(total=len(remaining), unit='ticker',
                  bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]') as pbar:
            for i, future in enumerate(as_completed(futures), 1):
                sym = futures[future]
                try:
                    row = future.result(timeout=30)
                except Exception as e:
                    row = {k: False for k in ['scan1','scan2','scan3','scan4','momentum',
                           'flag_ti65','flag_9mvol','flag_plus4_bo','flag_minus4_bd']}
                    row.update(ticker=sym, error=str(e)[:40])
                results.append(row)
                pbar.set_postfix({'last': sym}, refresh=False)
                pbar.update(1)
                if i % SB_BATCH_SIZE == 0:
                    sb_save_checkpoint(results, cp_path, hits_path)
                    elapsed = time.time() - start_time
                    eta_s   = (len(remaining) - i) / (i / elapsed) if elapsed else 0
                    pbar.set_postfix({'ETA': f'{int(eta_s//60)}m{int(eta_s%60)}s'})
                gc.collect()

    df = pd.DataFrame(results)
    df['any_hit'] = df[['scan1','scan2','scan3','scan4','momentum']].any(axis=1)
    df.to_csv(results_path, index=False)
    df[df['any_hit']].to_csv(hits_path, index=False)

    elapsed = time.time() - start_time
    print(f'\n{"━"*60}')
    print(f'  ✅ DONE  |  {elapsed/60:.1f} min  |  {int(df["any_hit"].sum())} hits')
    print(f'{"━"*60}\n')
    return df

print('✅ Stockbee runner ready')


In [ ]:
# Stockbee: Results Display

def sb_display_results(df):
    scan_defs = [
        ('scan1',    'SCAN 1',   'Extreme Sales Growth (99%+ YoY)'),
        ('scan2',    'SCAN 2',   'Sales Growth (39%/39% fail-closed 2Q)'),
        ('scan3',    'SCAN 3',   'Growth/Turnaround (<=11B USD, IPO <10y)'),
        ('scan4',    'SCAN 4',   'Earnings + Sales (39%E both Q / 20%S)'),
        ('momentum', 'MOMENTUM', 'ATR Entry Zone (MA70 + 2xATR + DI)'),
    ]
    cols = ['ticker','price','sales_pct','eps_pct','annual_sales_m',
            'mktcap_b','avg_vol_50k','ti65','adr_pct','ext_atr']

    total = int(df['any_hit'].sum()) if 'any_hit' in df.columns else 0
    print(f'\n{"="*80}')
    print(f'  STOCKBEE RESULTS  |  {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'  Universe: {len(df):,}  |  Hits: {total}')
    print(f'{"="*80}')

    for key, tag, label in scan_defs:
        if key not in df.columns: continue
        subset = df[df[key]].copy()
        if subset.empty:
            print(f'\n  [{tag}] {label}  →  No hits'); continue
        subset = subset.sort_values('sales_pct', ascending=False, na_position='last')
        subset['flags'] = (
            subset['flag_9mvol'].apply(lambda x: '🔊' if x else '') +
            subset['flag_plus4_bo'].apply(lambda x: '🚀' if x else '') +
            subset['flag_minus4_bd'].apply(lambda x: '🔻' if x else '') +
            subset['flag_ti65'].apply(lambda x: '📈' if x else '')
        )
        print(f'\n  ▌ [{tag}] {label}  ({len(subset)} stocks)')
        display_cols = [c for c in cols if c in subset.columns] + ['flags']
        try:
            from IPython.display import display as ipy_display
            ipy_display(subset[display_cols].reset_index(drop=True))
        except ImportError:
            print(subset[display_cols].to_string(index=False))

    print(f'\n  Flags: 🔊 9M+vol+4%up  🚀 +4% BO  🔻 -4% BD  📈 TI65 bullish\n')


def sb_top_momentum(df, n=20):
    """Top N stocks in ATR entry zone ranked by TI65."""
    if 'momentum' not in df.columns: return df.head(0)
    mom = df[df['momentum']].copy()
    mom['score'] = mom['ti65'].fillna(0)
    return mom.sort_values('score', ascending=False).head(n)

print('✅ Stockbee display functions ready')


In [ ]:
# Stockbee: ENTRY POINT
# @title ▶️ Run Stockbee Scan
# ⏱️ ~45-90 min for full universe. Checkpoint auto-saves every 50 tickers.

if True:
    save_dir = '.'

    # Run scan
    df = run_stockbee_scan(ALL_TICKERS, save_dir=save_dir)

    # Display results
    df['any_hit'] = df[['scan1','scan2','scan3','scan4','momentum']].any(axis=1)
    sb_display_results(df)

    # Top momentum
    print('\n  🏆 TOP 20 MOMENTUM STOCKS (ATR Entry Zone, ranked by TI65)\n')
    top = sb_top_momentum(df, 20)
    try:
        from IPython.display import display as ipy_display
        ipy_display(top[['ticker','price','ti65','ext_atr','adr_pct']].reset_index(drop=True))
    except Exception:
        print(top[['ticker','price','ti65','ext_atr']].to_string(index=False))

    print(f'\n  📁 Saved: {HITS_FILE} / {RESULTS_FILE}\n')


In [ ]:
# @title 📋 Stockbee Final Summary — Comma-Separated Ticker Lists

from IPython.display import display, HTML

def sb_print_summary(df):
    scan_defs = [
        ('scan1',    'SCAN 1',   'Extreme Sales Growth',  '99%+ YoY revenue',              '#f59e0b'),
        ('scan2',    'SCAN 2',   'Sales Growth',          '39%/39% fail-closed 2Q',         '#10b981'),
        ('scan3',    'SCAN 3',   'Growth / Turnaround',   '<=11B USD mktcap + IPO <10y',    '#a78bfa'),
        ('scan4',    'SCAN 4',   'Earnings + Sales',      '39%E both Q + 20%S fail-closed', '#00e5ff'),
        ('momentum', 'MOMENTUM','ATR Entry Zone',         'MA70 + 2xATR + DI',              '#ef4444'),
    ]
    scan_cols = [s[0] for s in scan_defs if s[0] in df.columns]
    all_hits  = sorted(df[df[scan_cols].any(axis=1)]['ticker'].dropna().unique().tolist())

    print('\n' + '='*70)
    print(f'  STOCKBEE SUMMARY  |  {datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'  Scanned: {len(df):,}  |  Unique hits: {len(all_hits)}')
    print('='*70)

    group_tickers = {}
    for key, tag, name, criteria, _ in scan_defs:
        if key not in df.columns: continue
        tickers = (
            df[df[key]].sort_values('sales_pct', ascending=False, na_position='last')
            ['ticker'].dropna().unique().tolist()
        )
        group_tickers[key] = tickers
        print(f'\n  [{tag}] {name} — {criteria}')
        print(f'  Count   : {len(tickers)}')
        print(f'  Tickers : {", ".join(tickers) if tickers else "(none)"}')

    for flag, emoji, label in [
        ('flag_9mvol',     '🔊', '9M Vol + price up 4%+ (EP signal)'),
        ('flag_plus4_bo',  '🚀', '+4% Breakout (vol confirmed)'),
        ('flag_minus4_bd', '🔻', '-4% Breakdown (vol confirmed)'),
        ('flag_ti65',      '📈', 'TI65 Bullish'),
    ]:
        if flag not in df.columns: continue
        fl = sorted(df[df[flag] & df[scan_cols].any(axis=1)]['ticker'].dropna().unique().tolist())
        print(f'\n  {emoji} {label} ({len(fl)}) : {", ".join(fl) if fl else "(none)"}')

    print(f'\n  ALL HITS (A-Z) [{len(all_hits)}]: {", ".join(all_hits)}')
    print('='*70)

    # HTML card
    rows_html = ''
    for key, tag, name, criteria, color in scan_defs:
        tickers = group_tickers.get(key, [])
        chips = ' '.join(
            f'<span style="background:{color}22;border:1px solid {color}66;color:{color};'
            f'padding:2px 8px;border-radius:4px;font-family:monospace;font-size:13px;'
            f'margin:2px;display:inline-block">{t}</span>' for t in tickers
        ) or '<span style="color:#555">No hits</span>'
        rows_html += f'''
        <tr>
          <td style="padding:10px 14px;border-bottom:1px solid #1e293b;white-space:nowrap;vertical-align:top">
            <span style="background:{color}22;border:1px solid {color}55;color:{color};
              padding:3px 9px;border-radius:4px;font-family:monospace;font-weight:bold;font-size:12px">{tag}</span>
            <div style="color:#94a3b8;font-size:12px;margin-top:5px">{name}</div>
            <div style="color:#475569;font-size:11px">{criteria}</div>
          </td>
          <td style="padding:10px 14px;border-bottom:1px solid #1e293b;text-align:center;
            font-family:monospace;font-size:16px;color:#e2e8f0;vertical-align:top">{len(tickers)}</td>
          <td style="padding:10px 14px;border-bottom:1px solid #1e293b;vertical-align:top">{chips}</td>
        </tr>'''

    all_chips = ' '.join(
        f'<span style="background:#1e293b;border:1px solid #334155;color:#cbd5e1;'
        f'padding:2px 8px;border-radius:4px;font-family:monospace;font-size:13px;'
        f'margin:2px;display:inline-block">{t}</span>' for t in all_hits
    ) or '<span style="color:#555">No hits</span>'

    display(HTML(f'''
    <div style="background:#0f172a;border-radius:12px;padding:24px;margin-top:16px;font-family:sans-serif">
      <div style="color:#00e5ff;font-family:monospace;font-size:20px;font-weight:700">📡 STOCKBEE — FINAL SUMMARY</div>
      <div style="color:#475569;font-size:12px;margin:6px 0 20px">
        {datetime.now().strftime("%Y-%m-%d %H:%M")} &nbsp;·&nbsp; {len(df):,} tickers &nbsp;·&nbsp;
        <b style="color:#94a3b8">{len(all_hits)} unique hits</b>
      </div>
      <table style="width:100%;border-collapse:collapse;border:1px solid #1e293b;border-radius:8px;overflow:hidden">
        <thead><tr style="background:#1e293b">
          <th style="padding:10px 14px;text-align:left;color:#475569;font-size:11px;letter-spacing:.1em;text-transform:uppercase;border-bottom:1px solid #0f172a">Scan</th>
          <th style="padding:10px 14px;text-align:center;color:#475569;font-size:11px;letter-spacing:.1em;text-transform:uppercase;border-bottom:1px solid #0f172a">#</th>
          <th style="padding:10px 14px;text-align:left;color:#475569;font-size:11px;letter-spacing:.1em;text-transform:uppercase;border-bottom:1px solid #0f172a">Tickers</th>
        </tr></thead>
        <tbody style="background:#0f172a">
          {rows_html}
          <tr style="background:#1e293b">
            <td style="padding:12px 14px;vertical-align:top"><span style="color:#e2e8f0;font-family:monospace;font-weight:700">ALL HITS</span><div style="color:#475569;font-size:11px;margin-top:3px">unique · A-Z</div></td>
            <td style="padding:12px 14px;text-align:center;font-family:monospace;font-size:18px;font-weight:700;color:#00e5ff;vertical-align:top">{len(all_hits)}</td>
            <td style="padding:12px 14px;vertical-align:top">{all_chips}</td>
          </tr>
        </tbody>
      </table>
    </div>
    '''))

sb_print_summary(df)


---
## Publish All Results to Google Sheets

In [ ]:
# PUBLISH — Write results to Google Sheets
import subprocess, sys
subprocess.check_call([sys.executable,"-m","pip","install","-q","gspread","gspread-dataframe"])
import gspread
from gspread_dataframe import set_with_dataframe
from google.colab import auth
from oauth2client.client import GoogleCredentials
import pandas as pd
from datetime import datetime

import google.auth
# ── Auth: the correct pattern for Colab 2024+ ────────────────
# Step 1: trigger Google sign-in
auth.authenticate_user()

# Step 2: get credentials using google-auth with explicit scopes
from google.auth import default
from google.auth.transport.requests import Request

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds, _ = default(scopes=SCOPES)

# Step 3: force a token refresh so it's valid
try:
    creds.refresh(Request())
except Exception:
    pass  # May already be valid

# Step 4: connect gspread
gc_client = gspread.authorize(creds)
sh = gc_client.open_by_key(SHEET_ID)
ts = datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Connected: {sh.title}")
print(f"URL: https://docs.google.com/spreadsheets/d/{SHEET_ID}")

def write_tab(df, name):
    name=name[:30]
    try: ws=sh.worksheet(name); ws.clear()
    except gspread.WorksheetNotFound: ws=sh.add_worksheet(title=name,rows=3000,cols=25)
    if not df.empty: set_with_dataframe(ws,df)
    print(f"  Written: {name} ({len(df)} rows)")

# Detect which hits dicts exist and build summary
all_labels = {
    "bearish_momentum":"Bearish Momentum","vcp_setup":"VCP Setup",
    "satish_bullish":"Satish Bullish","hardik":"Hardik Scan",
    "nr6":"NR6 Narrowest Range","rocket_base":"Rocket Base",
    "ep_breakout":"EP Breakout","high_20d":"20-Day High (US)",
    "high_2month":"2-Month High (US)","gapup_3pct":"Gap Up 3pct",
    "up_4pct":"Up 4pct Today","option_sell":"Option Sell (Chirag)",
    "rathod_bullish":"Rathod Intraday Bullish",
}
# Also Stockbee
sb_labels = {
    "scan1":"Stockbee Extreme Sales","scan2":"Stockbee Sales Growth",
    "scan3":"Stockbee Growth Turnaround","scan4":"Stockbee Earnings Sales",
    "momentum":"Stockbee ATR Momentum",
}

summary=[]
for k,label in all_labels.items():
    sp  = sp500_hits.get(k,[])  if "sp500_hits"  in dir() else []
    r2k = r2000_hits.get(k,[]) if "r2000_hits" in dir() else []
    nif = nifty_hits.get(k,[]) if "nifty_hits"  in dir() else []
    if not sp and not r2k and not nif: continue
    summary.append({"Scan":label,"SP500":len(sp),"R2K":len(r2k),"Nifty":len(nif),
        "Total":len(sp)+len(r2k)+len(nif),"SP500_tickers":", ".join(sp),
        "R2K_tickers":", ".join(r2k),"Nifty_tickers":", ".join(nif),"Updated":ts})
    rows=([{"Ticker":t,"Universe":"S&P 500","Updated":ts} for t in sp]+
          [{"Ticker":t,"Universe":"Russell 2000","Updated":ts} for t in r2k]+
          [{"Ticker":t,"Universe":"Nifty 500","Updated":ts} for t in nif])
    if rows: write_tab(pd.DataFrame(rows),label)

if "df" in dir() and isinstance(df,pd.DataFrame):
    for k,label in sb_labels.items():
        if k in df.columns:
            sub=df[df[k]].copy(); sub["Updated"]=ts
            if not sub.empty:
                summary.append({"Scan":label,"SP500":"","R2K":"","Nifty":"",
                    "Total":len(sub),"SP500_tickers":", ".join(sub.get("ticker",pd.Series()).tolist()[:50]),
                    "R2K_tickers":"","Nifty_tickers":"","Updated":ts})
                write_tab(sub,label)

if summary:
    write_tab(pd.DataFrame(summary),"Summary")

print(f"\nPublished {ts}")
print(f"Sheet: https://docs.google.com/spreadsheets/d/1rzc_6fZoHMFi1Ee75zRmIuGxeCWog1E62pZp9c1Lsfs")

In [ ]:
# ── SAVE TO GOOGLE DRIVE folder: scan_result ──────────────────────────
# Folder: https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c
# This cell runs independently — no drive.mount needed
import os, pandas as pd
from datetime import datetime
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaInMemoryUpload

auth.authenticate_user()
from google.auth import default
_creds, _ = default()
_svc = build('drive', 'v3', credentials=_creds)
_FOLDER_ID = "1qFAeut_82HvPP2-tntvxmxVl-uysms9c"

def upload_csv(df, filename):
    csv_bytes = df.to_csv(index=False).encode('utf-8')
    media = MediaInMemoryUpload(csv_bytes, mimetype='text/csv', resumable=False)
    q = "name='" + filename + "' and '" + _FOLDER_ID + "' in parents and trashed=false"
    existing = _svc.files().list(q=q, fields='files(id,name)').execute().get('files', [])
    if existing:
        _svc.files().update(fileId=existing[0]['id'], media_body=media).execute()
        print("  Updated:", filename, "->", len(df), "rows ->", "https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c")
    else:
        meta = dict(name=filename, parents=[_FOLDER_ID])
        _svc.files().create(body=meta, media_body=media, fields='id').execute()
        print("  Created:", filename, "->", len(df), "rows ->", "https://drive.google.com/drive/folders/1qFAeut_82HvPP2-tntvxmxVl-uysms9c")

_ts = datetime.now().strftime('%Y-%m-%d %H:%M')
_lmap = {
    'bearish_momentum':'Bearish Momentum','vcp_setup':'VCP Setup',
    'satish_bullish':'Satish Bullish','hardik':'Hardik Scan',
    'nr6':'NR6 Narrowest Range','rocket_base':'Rocket Base',
    'ep_breakout':'EP Breakout','high_20d':'20-Day High',
    'high_2month':'2-Month High','gapup_3pct':'Gap Up 3pct','up_4pct':'Up 4pct'
}
_rows = []
for _k in DAILY_SCAN_KEYS:
    _lbl = _lmap.get(_k, _k)
    for _t in sp500_hits.get(_k, []):  _rows.append({'Scan':_lbl,'Ticker':_t,'Universe':'S&P 500',    'Updated':_ts})
    for _t in r2000_hits.get(_k, []):  _rows.append({'Scan':_lbl,'Ticker':_t,'Universe':'Russell 2000','Updated':_ts})
    for _t in nifty_hits.get(_k, []):  _rows.append({'Scan':_lbl,'Ticker':_t,'Universe':'Nifty 500',  'Updated':_ts})
_df = pd.DataFrame(_rows) if _rows else pd.DataFrame(columns=['Scan','Ticker','Universe','Updated'])
upload_csv(_df, 'daily_hits.csv')
